# Notebook to accompany workshop 'From a mess to a map'

This workshop and notebook demonstrate some of the problems which may be encountered in preparing messy data so it can be examined using computational tools. The messy data we will work with is a set of responses to a question in an online survey - participants were asked to provide an Australian slang expression which they knew to mean 'Something very good'. The question of interest is whether there is variation by state in the proportion of the most popular responses. It is relatively easy to use computational tools to produce maps showing this sort of information, but the input to those tools has a particular, highly organised format.

We need to start out knowing what that final data format will look like. This is a very general rule: find out what data format is required by the tool you want to use before you start manipulating the data! In general, the mapping tool we will use (part of the ```ggplot2``` package in R) expects an input table where each geographic entity is a row and that row contains the geodata associated with the entity in one column and then any number of other columns representing attributes of the geographic entity which we are interested in. For the present exercise, this means a table with a row for each state of Australia which will contain the geodata and (minimally) the percentage response rate from that state for each of the five most popular responses.

Along the way, we will ignore some complications; I will try to at least acknowledge when this happens.

### Data Sources:

The ```data``` folder which is part of the notebook environment contains the two data files we will be using:

- the file with responses to the prompt 'What is an Australian slang expression for something very good?', Item01-VeryGood.csv
- the file with information about the respondents, ParticipantDemographics.csv

The ```data``` folder also contains a file, Slang_Data_Provenance.pdf, giving information about how the data was collected. The full data set is available from [this page](https://data.ldaca.edu.au/collection?id=arcp%3A%2F%2Fname%2Chdl10.26180~30102115&_crateId=arcp%3A%2F%2Fname%2Chdl10.26180~30102115)

### Code Preparation:
We will use a number of R packages. Several of these provide tidyverse functionality; the following ones are used for more specific purposes:
- ```stringr```: functions for working with strings
- ```ozmaps```: geographic data for making maps of Australia
- ```ggplot2```: a widely used plotting package
- ```viridis```: colour palettes for plotting
- ```patchwork```: to create arrays of plots
- ```repr```: to handle setting options

The following code cell installs the packages and then the second cell activates them. If you have the packages installed alreday, you can skip the first cell.

In [ ]:
## install packages if necessary
install.packages("dplyr")
install.packages("ggplot2")
install.packages("ozmaps")
install.packages("patchwork")
install.packages("purrr")
install.packages("readr")
install.packages("repr")
install.packages("stringr")
install.packages("tibble")
install.packages("tidyr")
install.packages("viridis")

In [ ]:
## load packages
library(dplyr)
library(ggplot2)
library(ozmaps)
library(patchwork)
library(purrr)
library(readr)
library(repr)
library(stringr)
library(tibble)
library(tidyr)
library(viridis)

# set plot options
options(repr.plot.width = 20, repr.plot.height = 12)

In [ ]:
## Load data files - fill in path to each file
responses <- read_csv('ro-crate/Item01-VeryGood.csv', show_col_types = FALSE)
participants <- read_csv('ro-crate/ParticipantDemographics.csv', show_col_types = FALSE)

In [ ]:
## let's see what we have
head(responses)

In [ ]:
head(participants)

The information which we need from the ```participants``` dataframe is the ID for each participant and their location when they completed the survey, ```custom.currentLocation```. First we extract this information to a new dataframe, then we get rid of any participants for whom we do not have a postcode. When you run the code in the second cell below, you will see a warning message ```NAs introduced by coercion``` - this is exactly what is intended. The function ```as.numeric``` tries to coerce values to be numbers and if there is text (or nothing) in the field, an NA value is generated. This makes it easy to filter out the participants without postcodes. You will see that we go from 2984 participants to 2695 as a result of this filtering.

**Complication**: In the survey, we asked people whether in their childhood they lived in a different place to where they lived when they participated in the survey. The idea was that some aspects of language are fixed quite early in life and we might at some point be interested in tracking that in our data. There are two reasons why this may not be as useful as we thought:
- vocabulary is part of language which stays flexible through life (compared to e.g. pronunciation)
- geographic variation in the data seems to be more about proportions of choices rather than categorical choices and therefore it would be very hard to identify childhood influence. 

In [ ]:
## extract participantID, postcode
participants_postcodes_1 <- participants |>
  select("participantID", "custom:currentLocation") |>
  rename(postCode = "custom:currentLocation")

str(participants_postcodes_1)

In [ ]:
## clean up list
participants_postcodes_2 <- participants_postcodes_1 |>
  mutate(postCode = as.numeric(postCode)) |>
  drop_na(postCode)

In [ ]:
nrow(participants_postcodes_2)

Now we can import the postcodes into the table of response data using the ```left_join``` function of R. We specify that we want to match the two data sources on the ```participantID``` value. As you can see, the new dataframe has all the data from the response table plus a postcode for each participant.

In [ ]:
responses_postcodes_3 <- left_join(responses, participants_postcodes_2, by = join_by(participantID))

In [ ]:
head(responses_postcodes_3)

Now let's have a look at a listing of responses by frequency. If we look at the first few rows here, for example the entries for *bonza* and variants, we start to see the mess! But do please look further down. What other messy things do you see in the data?

In [ ]:
## look at frequency table for responses
responses_postcodes_3 |> 
  count(slang, sort = TRUE)

It is intuitively clear that some responses are variants of others. But the relationship is not the same in each case. We need to make decisions about how (or even whether) we want to group responses.

Starting with the easy case, we can see that e.g. *grouse* and *Grouse* are counted separately. Is there any reason why we would not normalise case for counting purposes? This is very easy to do: R has a function ```tolower```. Note that the code adds a new-column to the dataframe. This means that there is a provenance trail within the dataframe and this could be important if we decided to export this version of the data. The revised frequency table shows that this change has reduced mess a little.

In [ ]:
## all lower case
responses_postcodes_4 <- responses_postcodes_3 |> 
  mutate(slang_lower = str_to_lower(slang))

In [ ]:
## look again
responses_postcodes_4 |> 
  count(slang_lower, sort = TRUE)

Two of the top three entries in the table point us to a slightly more complicated question: can we generalise over spelling variation? Is there any difference between *bonza* and *bonzer* (and a few other variations further down)? The answer to this may depend on what we are investigating. For example, we might want to find out whether the spelling variation is associated with age: do younger people prefer one spelling? But when we are counting the most popular responses, does it make sense to separate *bonza* and *bonzer*?

others to discuss:
- small variations: *ripper, cracker*
- *beauty* and variants
- is *you ....* different?
- cf *she.....* - are templates expressions?

The computational resource we can use to do normalisation across spelling variants is **regular expressions** or regex. Regex is a pattern matching language which allows us to find complex patterns in text data. In the code here, we only use a small part of the resources of regex:
- the pipe character | separates alternatives
- round brackets enclose set of possibilities
- \b indicates a word boundary, in the code we have ```\\b``` because the backslash is a reserved character and has to be **escaped**
Note that in some cases the regex does not specify alternatives, but because no final word boundary is included ```\\bripp``` will match any word which starts *ripp*, including *ripper* and *ripping*.

In [ ]:
## categorising - 
## add another column to the data frame

responses_postcodes_5 <- responses_postcodes_4 |> mutate(
    slang_grouped = case_when(
        str_detect(slang_lower, "bonz(a|er)") ~ "bonza",
        str_detect(slang_lower, "\\bbe(aut|ut|aud|w)") ~ "beauty",
        str_detect(slang_lower, "\\bgrous") ~ "grouse",
        str_detect(slang_lower, "\\bripp") ~ "ripper",
        str_detect(slang_lower, "\\bcrack") ~ "cracker",
        .default = slang_lower
    )
)

## Look again
responses_postcodes_5 |> 
  count(slang_grouped, sort = TRUE)

If you look at the frequency table in detail, you will see that we haven't captured everything. For example, there is a *bonser* down there, and if you want to explore regex further, you could try changing the code to capture this variant as well. But there is a clear gap between the fifth most popular response and the sixth as well as clear gaps between each of the top five responses. A more exhaustive analysis of the variants might change the numbers slightly but it is very unlikely that the overall picture will change. For current purposes, this is good enough. 

**Complication**: But before we move on to getting counts by states, let's explore a little the problem of multi-word answers in this data. In some cases, respondents have provided a multi-word expression (e.g. *you little beauty*, in other cases, they have provided alternative answers (e.g. *beauty, bonza*). The regex above will ignore that difference and assigns a response to the group which is matched first from the five alternatives (if there is any match). What features of these resposnses might we be able to use to at least see how complex the problem is?

Some respondents have been helpful and used the word *or* to separate alternative answers. On the assumption that *or* is very unlikely to occur as part of a response expression, we can extract all of these examples and examine them.

**NB**: quotation marks/apostrophes are reserved characters, so they are escaped in the output which follows.

In [ ]:
## includes 'or'
responses_or <- subset(responses_postcodes, str_detect(responses_postcodes$slang_lower, '\\bor\\b'))
responses_or$slang_lower

This approach is picking up a lot of listing of responses. But there are still complications! There is one example where the *or* is used to separate two glosses (*bonza meaning good or nice*) and there are responses which include more than two expressions, with the position of *or* being unpredictable (*bonza beaut or beauty*, *bonza or rippa (ripper). deadly.*)

We can also make the assumption that response expressions are very unlikely to include commas or semi-colons and that responses which include those punctuation symbols include alternative responses.

In [ ]:
## includes comma or semi-colon
responses_comma <- subset(responses_postcodes, str_detect(responses_postcodes$slang_lower, ';|,'))
responses_comma$slang_lower

Again, it looks as though this approach is quite good for identifying alternative responses (but there is one example which contradicts my initial assumption about punctuation: *winner, winner chicken dinner*).

Using both *or* and punctuation, we could make a function which would parse out the alternative responses and add new lines to the dataframe consisting of a single response item and a copy of all the other fields associated with the original response. We can see that this would not be completely accurate and this raises a question which often arises in cleaning messy data: how much effort is it worth expending to catch one or two anomalies?

Approaching the problem from the opposite direction, we can make the assumption that multi-word responses which do **not** include *or* or relevant punctuation do consist of multi-word expressions.

In [ ]:
## includes space but not 'or', comma, semi-colon
response_multi <- subset(responses_postcodes, str_detect(responses_postcodes$slang_lower, ' ') & !str_detect(responses_postcodes$slang_lower, ';|,|or'))
response_multi$slang_lower

Again, this approach seems to work quite well, but also again, there are anomalies. Is *bonzer ripper grouse* a single expression, or did the respondent just not bother to indicate that these are alternatives? There are examples which are a single word plus some explanation, e.g. *beaudy (beauty with a \'d\' replacing the \'t\')*. And there are examples with dividers between alternatives which we haven't thought about: *beauty/ ripper*. Which all brings us back to the question raised previously: how much effort is it worth expending to catch these anomalies?

More positively, I hope the preceding discussion has shown that making plausible (sensible) assumptions and using simple computational approaches based on those assumptions can at least give us a good picture of complex, messy data.

Moving on, now we can get counts by state for the most popular responses. We do this by extracting subsets of responses depending on the postcodes and therefore we need to start by setting up a dataframe which has the ranges of postcodes for the different states (I'm treating ACT as part of NSW here....). 

In [ ]:
## categorise postcodes into states/territories

state_responses <- responses_postcodes_5 |>
  mutate(state = case_when(
      postCode >= 2000 & postCode <= 2999 ~ 'NSW', 
      postCode >= 3000 & postCode <= 3999 ~ 'VIC', 
      postCode >= 4000 & postCode <= 4999 ~ 'QLD', 
      postCode >= 5000 & postCode <= 5999 ~ 'SA', 
      postCode >= 6000 & postCode <= 6999 ~ 'WA', 
      postCode >= 7000 & postCode <= 7999 ~ 'TAS', 
      postCode >= 0800 & postCode <= 0899 ~ 'NT', 
      is.na(postCode) ~ NA
  )
        )

## calculate total responses for each state/territory
state_totals <- state_responses |>
  count(slang_grouped = "Total", state) |>
  filter(state %in% top5_cols) |>
  pivot_wider(names_from = slang_grouped, values_from = n)

state_totals

Remember that the data format we want has each geographic entity (here, the states) as a row in the table.

In [ ]:
## filter for the top 5 responses
top5_cols <- c('bonza', 'beauty', 'grouse', 'ripper', 'cracker')
top5_rows <- c('NSW', 'VIC', 'QLD', 'SA', 'WA', 'TAS', 'NT')

top5 <- state_responses |>
  filter(slang_grouped %in% top5_cols,
        state %in% top5_rows) |>
  mutate(slang_grouped = factor(slang_grouped, levels = top5_cols), 
         state = factor(state, levels = top5_rows)
        ) |> # ensures results are organised by the intended order above
  count(state, slang_grouped, .drop = FALSE) |>
  pivot_wider(names_from = slang_grouped, values_from = n)

## add total col
top5_with_total <- top5 |>
  left_join(state_totals, join_by(state))

top5_with_total

We also add a column showing the percentage of responses for each state for each of the five responses. We do this because comparing raw counts won't tell us much except that there are more responses from the more populous states.

In [ ]:
top5_with_percent <- top5_with_total |>
  mutate(
      across(c(bonza:cracker), ~ .x/Total * 100, .names = "{.col}_percent")
  ) |>
  rename_with(~ paste0(.x, "_raw", recycle0 = TRUE), c(bonza:cracker))

top5_with_percent                 

Now we can put together the table which will be the input for making maps. The ```ozmaps``` package provides a list of polygon data for the states of Australia. We discard ACT (see above) and Other Territories and then add our data about the top five responses by state as additional columns in the table.

In [ ]:
## putting mapping data together

## discard ACT and Other Territories
state_geos <- ozmap_data('states') |>
  slice(-(8:9))

# add top 5 responses by state
state_geos_with_responses <- bind_cols(state_geos, top5_with_percent)

In [ ]:
str(state_geos_with_responses)

I want to make an array of maps so that it is easy to compare the geographic distribution of the different responses. To do this, rather than immediately displaying the maps, I assign them to plot objects which can then be used by the ```patchwork``` package to make the nice array. (```p6``` is an empty plot used to make all the other plots equal in size.)

In [ ]:
# function to create plot object
create_plot_object <- function(top5, slang_term, state_geos) {
    col_name <- str_c(slang_term, '_percent')
    slang_col <- top5 |>
      pull(col_name)
    
    legend <- str_c('% responses\n"something very good" = "', slang_term, '"')
    plot <- ggplot(state_geos) + 
      geom_sf(aes(fill = slang_col)) + 
      scale_color_identity() + 
      scale_fill_viridis(option = 'F', begin = 1, end = 0.4) + 
      labs(fill = legend)
    
    return(plot)
}

plot_names <- paste0("p", 1:5)
slang_terms <- top5_cols

# create 5 plot objects
plots <- map(slang_terms, \(st) create_plot_object(top5_with_percent, st, state_geos_with_responses)) |>
  set_names(plot_names)

# create 6th (empty) plot object
p6 <- ggplot() + 
  theme(panel.background = element_rect(fill = "white", colour = "white"), 
        plot.background = element_rect(fill = "white", colour = "white"))

In [ ]:
## use patchwork to create array of plots

(plots$p1 + plots$p2 + plots$p3) / (plots$p4 + plots$p5 + p6)

One last question to think about: the plots we are looking at use colour to show proportions for each of the five responses separately so the colours cannot be compared across plots. We can produce a version which has comparable colours across plots by setting the limits of the scale to be the same in each case - the following code cell produces this alternative. Which version makes more sense to you?

In [ ]:
# function to create plot object
create_plot_object <- function(top5, slang_term, state_geos) {
    col_name <- str_c(slang_term, '_percent')
    slang_col <- top5 |>
      pull(col_name)
    
    legend <- str_c('% responses\n"something very good" = "', slang_term, '"')
    plot <- ggplot(state_geos) + 
      geom_sf(aes(fill = slang_col)) + 
      scale_color_identity() + 
      scale_fill_viridis(option = 'F', begin = 1, end = 0.4, limits=c(0, 60)) + 
      labs(fill = legend)
    
    return(plot)
}

plot_names <- paste0("p", 1:5)
slang_terms <- top5_cols

# create 5 plot objects
plots <- map(slang_terms, \(st) create_plot_object(top5_with_percent, st, state_geos_with_responses)) |>
  set_names(plot_names)

# create 6th (empty) plot object
p6 <- ggplot() + 
  theme(panel.background = element_rect(fill = "white", colour = "white"), 
        plot.background = element_rect(fill = "white", colour = "white"))

## use patchwork to create array of plots

(plots$p1 + plots$p2 + plots$p3) / (plots$p4 + plots$p5 + p6)

And there you have it... from a mess to a map!